# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [1]:
ROLL_NO = 102417332          # <-- your full roll number
NAME = "Aditya"           # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [3]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [4]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 102417332   Name: Aditya   Fingerprint: B7FEFA87   Tickets: 14

Ticket_ID                                                                          Ticket_Text
      T01                  I tried restarting the email while uploading the file. Please help.
      T02                Wifi keeps disconnecting before the assignment deadline. Please help.
      T03          I reinstalled the server when connected to campus Wi-Fi. Please resolve it.
      T04   Password is running slowly after restarting the system. Is there another solution?
      T05                                                          Ticket #4521 is still open.
      T06       I was locked out of the projector after resetting my password. This is urgent.
      T07        E-mail failed to open after restarting the system. I don't know what changed.
      T08                                                        Version 2.0.1 crashes at 99%.
      T09 Printer failed to open when connected to campus Wi-Fi. It 

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [9]:
# Your code
# Q1. Preprocessing

import pandas as pd

results = []

for _, row in df.iterrows():
    output = process_ticket(row["Ticket_Text"])

    results.append({
        "Ticket_ID": row["Ticket_ID"],
        "Number of Sentences": len(output["sentences"]),
        "Number of Tokens": len(output["tokens"]),
        "Number of Punctuation Tokens": len(output["punct"]),
        "Number of Stopwords": len(output["stops"]),
        "Number of Unique Tokens": len(set(output["tokens"])),
        "Number of Final Tokens": len(output["final"])
    })

# Create preprocessing table
result_table = pd.DataFrame(results)

# Add total row for count-based columns
count_columns = [
    "Number of Sentences",
    "Number of Tokens",
    "Number of Punctuation Tokens",
    "Number of Stopwords",
    "Number of Final Tokens"
]

total_row = {"Ticket_ID": "TOTAL"}

for column in count_columns:
    total_row[column] = result_table[column].sum()

total_row["Number of Unique Tokens"] = "-"

result_table = pd.concat(
    [result_table, pd.DataFrame([total_row])],
    ignore_index=True
)

print("PREPROCESSING TABLE")
display(result_table)

# Display actual token lists for selected ticket T01
selected = df[df["Ticket_ID"] == "T01"].iloc[0]
details = process_ticket(selected["Ticket_Text"])

print("\nSELECTED TICKET:", selected["Ticket_ID"])
print("Ticket Text:", selected["Ticket_Text"])
print("Actual Tokens:", details["tokens"])
print("Punctuation Tokens:", details["punct"])
print("Stopwords:", details["stops"])
print("Final Tokens:", details["final"])


PREPROCESSING TABLE


,Ticket_ID,Number of Sentences,Number of Tokens,Number of Punctuation Tokens,Number of Stopwords,Number of Unique Tokens,Number of Final Tokens
0,T01,2,13,2,4,11,7
1,T02,2,11,2,2,10,7
2,T03,2,14,2,5,13,7
3,T04,2,14,2,5,13,7
4,T05,1,7,2,1,7,4
5,T06,2,16,2,9,15,5
6,T07,2,16,2,6,15,8
7,T08,1,7,2,1,7,4
8,T09,2,16,2,5,14,9
9,T10,2,12,2,6,12,4



SELECTED TICKET: T01
Ticket Text: I tried restarting the email while uploading the file. Please help.
Actual Tokens: ['i', 'tried', 'restarting', 'the', 'email', 'while', 'uploading', 'the', 'file', '.', 'please', 'help', '.']
Punctuation Tokens: ['.', '.']
Stopwords: ['i', 'the', 'while', 'the']
Final Tokens: ['tried', 'restarting', 'email', 'uploading', 'file', 'please', 'help']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [10]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Your code
# Q2. Regex Extraction and Replacement

import re

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Regex patterns
patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers (including decimals and versions)": r"(?<![\w.])\d+(?:\.\d+)+(?![\w.])|(?<![\w.])\d+(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words beginning with a vowel": r"\b[AEIOUaeiou][A-Za-z]*\b"
}

# Replacement patterns
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"https?://[^\s]+|www\.[^\s]+"
phone_pattern = r"(?<!\w)(?:\+91[\s-]?\d{5}[\s-]?\d{5}|0\d{3}[-\s]\d{3}[-\s]\d{4})(?!\w)"

# Combine raw tickets and test lines
all_texts = list(df["Ticket_Text"]) + TEST_LINES

# Extract matches from every raw ticket and test line
for i, text in enumerate(all_texts, start=1):
    print(f"\nTEXT {i}: {text}")

    for label, pattern in patterns.items():
        print(f"{label}: {re.findall(pattern, text)}")

# Test replacements on the supplied TEST_LINES
print("\n\nREGEX REPLACEMENT RESULTS")

for line in TEST_LINES:
    result = re.sub(email_pattern, "<EMAIL>", line)
    result = re.sub(url_pattern, "<URL>", result)
    result = re.sub(phone_pattern, "<PHONE>", result)

    print("\nOriginal:", line)
    print("Replaced:", result)



TEXT 1: I tried restarting the email while uploading the file. Please help.
Words longer than 5 letters: ['restarting', 'uploading', 'Please']
Numbers (including decimals and versions): []
Capitalized words: ['I', 'Please']
Words beginning with a vowel: ['I', 'email', 'uploading']

TEXT 2: Wifi keeps disconnecting before the assignment deadline. Please help.
Words longer than 5 letters: ['disconnecting', 'before', 'assignment', 'deadline', 'Please']
Numbers (including decimals and versions): []
Capitalized words: ['Wifi', 'Please']
Words beginning with a vowel: ['assignment']

TEXT 3: I reinstalled the server when connected to campus Wi-Fi. Please resolve it.
Words longer than 5 letters: ['reinstalled', 'server', 'connected', 'campus', 'Please', 'resolve']
Numbers (including decimals and versions): []
Capitalized words: ['I', 'Wi', 'Fi', 'Please']
Words beginning with a vowel: ['I', 'it']

TEXT 4: Password is running slowly after restarting the system. Is there another solution?
Words

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [11]:
# Your code
# Q3. Custom Tokenizer vs word_tokenize

import re
from nltk.tokenize import RegexpTokenizer, word_tokenize

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# One regex pattern to preserve URLs, emails, phone numbers,
# versions/decimals, hyphenated words, contractions, and words.
pattern = (
    r"https?://[^\s]+|www\.[^\s]+|"
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|"
    r"\+91[\s-]?\d{5}[\s-]?\d{5}|"
    r"\b0\d{3}[-\s]\d{3}[-\s]\d{4}\b|"
    r"\b\d+(?:\.\d+)+\b|\b\d{1,3}(?:,\d{3})+\b|\b\d+\b|"
    r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b|"
    r"\b[A-Za-z]+(?:'[A-Za-z]+)+\b|"
    r"\b[A-Za-z]+\b|[^\w\s]"
)

my_tokenize = RegexpTokenizer(pattern).tokenize

# Run on TEST_LINES
print("========== TEST_LINES ==========")

for i, text in enumerate(TEST_LINES, start=1):
    print(f"\nLine {i}: {text}")
    print("my_tokenize:  ", my_tokenize(text))
    print("word_tokenize:", word_tokenize(text))

# Run on the generated tickets and compare token lists
print("\n========== GENERATED TICKETS ==========")

different_count = 0

for _, row in df.iterrows():
    text = row["Ticket_Text"]

    custom_tokens = my_tokenize(text)
    nltk_tokens = word_tokenize(text)

    different = custom_tokens != nltk_tokens

    if different:
        different_count += 1

    print(f"\n{row['Ticket_ID']}: {text}")
    print("my_tokenize:  ", custom_tokens)
    print("word_tokenize:", nltk_tokens)
    print("Different:", different)

print("\n========== OBSERVATION ==========")
print(
    "Number of tickets tokenized differently:",
    different_count,
    "out of",
    len(df)
)


========== TEST_LINES ==========

Line 1: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
my_tokenize:   ['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']
word_tokenize: ['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']

Line 2: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
my_tokenize:   ['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']
word_tokenize: ['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']

Line 3: The state-of-the-art lab isn't open; log-

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

2

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [12]:
# Your code
# Q4. Stemming and Lemmatization

import pandas as pd
import nltk
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

# Download required NLTK resources
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("omw-1.4", quiet=True)

porter = PorterStemmer()
lancaster = LancasterStemmer()

# Chosen suffix rule after inspecting ticket vocabulary:
# Remove common suffixes such as ing, ed, and s.
regex_stemmer = RegexpStemmer(r"(ing|ed|s)$", min=4)

lemmatizer = WordNetLemmatizer()

# Collect unique final tokens from all tickets
unique_final_tokens = set()

for text in df["Ticket_Text"]:
    output = process_ticket(text)
    unique_final_tokens.update(output["final"])

# POS tag each word and map Penn Treebank tags to WordNet POS
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    elif tag.startswith("V"):
        return "v"
    elif tag.startswith("N"):
        return "n"
    elif tag.startswith("R"):
        return "r"
    return "n"

tagged_words = nltk.pos_tag(sorted(unique_final_tokens))

results = []

for word, tag in tagged_words:
    p = porter.stem(word)
    l = lancaster.stem(word)
    r = regex_stemmer.stem(word)
    wn_pos = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos=wn_pos)

    # Show only words where at least one output differs
    if len({p, l, r, lemma}) > 1:
        results.append({
            "Original Word": word,
            "POS Tag": tag,
            "Porter Stem": p,
            "Lancaster Stem": l,
            "Regexp Stem": r,
            "WordNet Lemma": lemma
        })

comparison_table = pd.DataFrame(results)
display(comparison_table)


,Original Word,POS Tag,Porter Stem,Lancaster Stem,Regexp Stem,WordNet Lemma
0,another,DT,anoth,anoth,another,another
1,assignment,NN,assign,assign,assignment,assignment
2,campus,NN,campu,camp,campu,campus
3,changed,VBD,chang,chang,chang,change
4,computer,NN,comput,comput,computer,computer
5,crashes,NNS,crash,crash,crashe,crash
6,deadline,JJ,deadlin,deadlin,deadline,deadline
7,error,NN,error,er,error,error
8,file,NN,file,fil,file,file
9,nobody,NN,nobodi,nobody,nobody,nobody


**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

**Over-stemming:** The word **“error”** is reduced to **“er”** by LancasterStemmer. This removes too much of the original word and makes the stem less meaningful.

**Under-stemming:** The words **“changed”** and **“change”** are related word forms, but the RegexpStemmer produces **“chang”** for “changed” while leaving “change” unchanged. Thus, it fails to reduce both forms to the same stem.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Your code
# Q5. Corpus Statistics

import math
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.stem import PorterStemmer

porter = PorterStemmer()

S1 = []
S2 = []
S3 = []

# Process every generated ticket
for text in df["Ticket_Text"]:
    output = process_ticket(text)

    S1.extend(output["tokens"])
    S2.extend([token for token in output["tokens"]
               if not is_punct(token)])
    S3.extend(output["final"])

# S4: Porter-stemmed final tokens
S4 = [porter.stem(token) for token in S3]

# Corpus statistics
def corpus_stats(name, tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0

    return {
        "Corpus": name,
        "Total Tokens": total,
        "Unique Tokens": unique,
        "TTR": round(ttr, 4)
    }

stats = pd.DataFrame([
    corpus_stats("S1 - All tokens", S1),
    corpus_stats("S2 - No punctuation", S2),
    corpus_stats("S3 - Final tokens", S3),
    corpus_stats("S4 - Porter stems", S4)
])

print("CORPUS STATISTICS")
display(stats)

# Top 10 words in S1 and S3
print("\nTOP 10 WORDS IN S1")
print(pd.DataFrame(Counter(S1).most_common(10),
                   columns=["Word", "Frequency"]).to_string(index=False))

print("\nTOP 10 WORDS IN S3")
print(pd.DataFrame(Counter(S3).most_common(10),
                   columns=["Word", "Frequency"]).to_string(index=False))

# Log(rank) vs log(frequency) plot for S3
frequencies = sorted(Counter(S3).values(), reverse=True)
ranks = list(range(1, len(frequencies) + 1))

plt.figure(figsize=(8, 5))
plt.plot([math.log(r) for r in ranks],
         [math.log(f) for f in frequencies],
         marker="o", markersize=3)
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Log(Rank) vs Log(Frequency) for S3")
plt.grid(True)
plt.show()

# Top 5 bigrams
def top_bigrams(tokens, n=5):
    bigrams = zip(tokens, tokens[1:])
    return Counter(bigrams).most_common(n)

print("\nTOP 5 BIGRAMS IN S1")
for bigram, frequency in top_bigrams(S1):
    print(bigram, ":", frequency)

print("\nTOP 5 BIGRAMS IN S3")
for bigram, frequency in top_bigrams(S3):
    print(bigram, ":", frequency)


**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*

TTR (Type-Token Ratio) is the ratio of unique tokens to total tokens. S1 includes all tokens, including punctuation, while S4 contains the Porter-stemmed final tokens after preprocessing removes stopwords and punctuation. Therefore, the total number of tokens and the number of unique tokens change. Stemming combines some related word forms into the same stem, reducing vocabulary diversity. As a result, TTR generally decreases from S1 to S4 in my data, although the exact change depends on the token counts.